# 04b · One scale for everything: the 4 sequencing facilities (RGM) + the JDM lab

**Run order:** 02 → 03 → 04 → **04b (this notebook)** → 05 → 06 → 07. Kernel: **R (WGCNA)** (any R kernel that has `limma`, `edgeR`, `sva`).

## The problem, in your own data's terms

Your samples come from **seven "sources"**: the four facility files (BCM-HGSC, NCI-FNL, NIAID-NCI, NIAMS, and NIAMS is split into its sequencing cohorts) and the **JDM lab**. Each source nudges every gene up or down by a little, in its own way. If you simply put everything in one table, the biggest pattern you see is *which source a sample came from*, not which disease it has.

## How a shift can be measured, and when it cannot

A source's shift can only be measured from samples that exist **in more than one source**. Healthy controls (HC) are the ruler: the same kind of person, sequenced in several places. If the BCM-HGSC file holds CANDLE and HC, and the NCI-FNL file holds SAVI and HC, the HC tell us how BCM-HGSC differs from NCI-FNL. If a source holds **one** diagnosis that appears nowhere else, "this source" and "this disease" are the same thing and **no method can separate them**. (Anne's repo hit exactly this wall: notebook 49 tried to correct the labs with control genes and gave up, because each lab held one kind of sample.)

So this notebook does four things, in this order, and prints the answer to each:

1. **Source × diagnosis table**: who is in which source.
2. **Verdict per source**: `OK` (has healthy controls or a disease that another source also has) or `CANNOT BE CORRECTED` (nothing to measure its shift against). Sources that cannot be corrected are left out of the combined table and listed, never silently included.
3. **Correct** with ComBat on log2 values, with **diagnosis protected** so real disease differences are kept.
4. **Prove it worked**: (a) how much of the main patterns is "source" before and after, (b) the interferon genes must still be high in the diseases (a correction that flattens them has destroyed the signal), (c) where one disease sits in two sources, its fingerprint must agree between them.

## Why two steps (04, then here)

Notebook 04 corrected the facilities with **ComBat-seq on counts**, the proper method for counts. The JDM lab file is **RPKM, not counts**, so ComBat-seq cannot use it. Here the RGM side goes to log2 counts-per-million, the JDM side to log2(RPKM+1) (quantile-normalised as that lab did), they are joined by **gene symbol**, and ComBat is run once more with all sources as batches. A gene's different scale in RPKM vs CPM is just another per-gene shift, which ComBat removes the same way.

## What the result may be used for

**Plots, clustering, and as an input to downstream views.** Statistical tests still use each lab's own comparison, or raw counts with a batch term (notebook 05).

**Reads:** `rnaseq_out_v2/` files from 03 and 04; `/mnt/file-systems/4b_rpkm.csv` (its `Cohort` row says JDM or healthy; no key sheet needed); `genes/ifn-response-genes-28.txt`, `genes/nfkb-only-control-11-dejesus2020.txt`.
**Writes (all CSV, plots inline):** `rnaseq_out_v2/all_sources_correction/`
- `corrected_log2_all_sources.csv`: genes (symbol) × samples, the corrected values
- `corrected_samples.csv`: for each column, its diagnosis, source and lab
- `source_by_diagnosis.csv`, `source_verdicts.csv`: the table and the verdicts
- `excluded_samples.csv`: every sample left out and why
- `variance_explained_by_stage.csv`, `checks_signal_genes.csv`, `checks_shared_disease.csv`: the proof
- `PROCESSING_LOG_all_sources.md`

---
# Part 1 · Settings
Every decision is a variable. The defaults are the cautious ones.

| variable | default | meaning |
|---|---|---|
| `USE_ONE_PER_PATIENT` | `TRUE` | each person counted once (their chosen sample from 03). `FALSE` = all eligible samples |
| `MIN_PER_SOURCE` | `3` | a source with fewer samples than this is left out (too few to measure a shift) |
| `MIN_ANCHOR` | `3` | a diagnosis counts as "shared" between two sources only if each has at least this many |
| `IFN_MIN_SIGNAL` | `0.5` | the interferon check only judges a disease whose own-source effect (in z-score units) is at least this |
| `PASS_FRACTION` | `0.5` | after correction the interferon effect must keep at least this share of its own-source size |
| `REF_BATCH` | `NULL` | name of a source to leave untouched (all others are moved to match it). `NULL` = everyone moves to the common middle |

In [ ]:
RESULTS <- "/home/jovyan/session_data/Attempt0/endotype_results"
IN_DIR  <- file.path(RESULTS, "rnaseq_out_v2")
BC_DIR  <- file.path(IN_DIR, "batch_correction")            # written by notebook 04
OUT_DIR <- file.path(IN_DIR, "all_sources_correction")      # this notebook writes here
dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)

MNT      <- "/mnt/file-systems/"
JDM_RPKM <- paste0(MNT, "4b_rpkm.csv")                      # rows Cohort, ID, then genes by symbol (RPKM)
JDM_KEY  <- ""                                              # OPTIONAL key sheet; only used to spot JDM people already in the RGM set. "" = not used
HC_PATTERN  <- "^(hc|relative-hc|healthy|control)"          # a label (Cohort row of 4b_rpkm.csv) matching this = healthy control
JDM_PATTERN <- "jdm|dermatomyositis"                        # a label matching this = JDM
GENE_SET_DIR <- "/home/jovyan/session_data/endotype-transcripts-007/genes"

# ---- decisions you can flip ------------------------------------------------
USE_ONE_PER_PATIENT <- TRUE
MIN_PER_SOURCE      <- 3
MIN_ANCHOR          <- 3
ANCHOR_ON           <- "HC"    # which diagnoses may serve as the shared ruler between sources. "HC" = healthy controls only (strict). NULL = any diagnosis found in 2+ sources
JDM_SPLIT_BY_COHORT <- FALSE   # TRUE = treat each label in the Cohort row of 4b_rpkm.csv as its own source (use if the JDM file is really several datasets; check the label table in Part 3 first)
IFN_MIN_SIGNAL      <- 0.5
PASS_FRACTION       <- 0.5
QUANTILE_NORMALISE  <- TRUE      # JDM side: after log2(RPKM + 1), quantile-normalise (that lab's own recipe)
DROP_SAME_PERSON    <- TRUE      # drop JDM-lab samples of a person already in the RGM set
REF_BATCH           <- NULL
RECODE <- c("Undifferentiated Interferonopathy" = "UAID")

need_files <- c(JDM_RPKM,
                file.path(BC_DIR, "counts_combatseq_eligible.csv"),
                file.path(IN_DIR, "rnaseq_counts_raw_eligible.csv"),
                file.path(IN_DIR, "rnaseq_samples_eligible.csv"))
if (nzchar(JDM_KEY)) need_files <- c(need_files, JDM_KEY)
absent_files <- need_files[!file.exists(need_files)]
if (length(absent_files))
  stop("Missing file(s):\n  ", paste(absent_files, collapse = "\n  "),
       "\nNotebooks 02, 03 and 04 must have run to the end first (and the JDM files must be on /mnt/file-systems).")

need <- c("edgeR", "sva")
absent <- need[!vapply(need, requireNamespace, logical(1), quietly = TRUE)]
if (length(absent)) {                       # one-time install if the R environment lacks them
  if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager", repos = "https://cloud.r-project.org")
  BiocManager::install(absent, ask = FALSE, update = FALSE)
}
suppressMessages({library(limma); library(edgeR); library(sva)})

# ---- small helpers ---------------------------------------------------------
norm_id <- function(x) {                    # O-6033, o6033 and O6033 compare equal; CHLD0047 == LD0047
  x <- toupper(trimws(as.character(x))); x[is.na(x)] <- ""; x <- sub("\\.0$", "", x)
  x <- gsub("[-_ ]", "", x); x <- sub("^CHLD", "LD", x); x[x %in% c("NAN", "NONE", "NA")] <- ""; x }
symbol_rows <- function(M, sym) {           # one row per gene symbol: the best-expressed copy
  o <- order(rowMeans(M, na.rm = TRUE), decreasing = TRUE)
  o[!is.na(sym[o]) & sym[o] != "" & !duplicated(sym[o])] }
read_set <- function(f) { p <- file.path(GENE_SET_DIR, f); if (!file.exists(p)) return(NULL)
  x <- trimws(readLines(p, warn = FALSE)); x[x != "" & !startsWith(x, "#")] }
r2_of <- function(p, f, k = 3) sapply(seq_len(k), function(i) round(summary(lm(p$x[, i] ~ factor(f)))$r.squared, 2))
cat("settings OK; output folder:", OUT_DIR, "\n")

---
# Part 2 · The RGM samples (from 03 and 04)
Two versions of the same samples are read, because the proof in Part 6 needs both:
- **`nih0`**: the **raw** counts as log2 counts-per-million (no correction at all);
- **`nih1`**: the **ComBat-seq** counts of notebook 04 as log2 counts-per-million (facilities already corrected).

Genes that are barely expressed are removed first (`filterByExpr`), then each gene symbol is kept once (the best-expressed copy). The diagnosis `Undifferentiated Interferonopathy` is renamed `UAID`, as in 05.

In [ ]:
cs <- read.csv(file.path(BC_DIR, "counts_combatseq_eligible.csv"), check.names = FALSE)
cs_id  <- cs$Gene_ID; cs_sym <- cs$Gene_Name
CS <- as.matrix(cs[, !(colnames(cs) %in% c("Gene_ID", "Gene_Name"))]); storage.mode(CS) <- "numeric"
rownames(CS) <- cs_id

rw <- read.csv(file.path(IN_DIR, "rnaseq_counts_raw_eligible.csv"), check.names = FALSE, row.names = 1)
RW <- as.matrix(rw[, colnames(rw) != "Gene_Name"]); storage.mode(RW) <- "numeric"
rownames(RW) <- sub("\\.[0-9]+$", "", rownames(RW))                 # ENSG00000000003.10 -> ENSG00000000003
stopifnot(!anyNA(match(cs_id, rownames(RW))), !anyNA(match(colnames(CS), colnames(RW))))
RW <- RW[cs_id, colnames(CS)]                                       # same genes, same sample order as the corrected counts

s_all <- read.csv(file.path(IN_DIR, "rnaseq_samples_eligible.csv"), check.names = FALSE, colClasses = "character")
stopifnot(!anyNA(match(colnames(CS), s_all$sample_id)))
s_all <- s_all[match(colnames(CS), s_all$sample_id), ]
dx <- s_all$dx_clean; dx[dx == ""] <- "(blank)"
hit <- dx %in% names(RECODE); dx[hit] <- unname(RECODE[dx[hit]]); s_all$diagnosis <- dx

use <- if (USE_ONE_PER_PATIENT) s_all$one_per_patient == "True" else rep(TRUE, nrow(s_all))
s  <- s_all[use, ]; CS <- CS[, use, drop = FALSE]; RW <- RW[, use, drop = FALSE]

expressed <- filterByExpr(DGEList(RW), group = s$diagnosis)
CS <- CS[expressed, , drop = FALSE]; RW <- RW[expressed, , drop = FALSE]; cs_sym <- cs_sym[expressed]
nih1 <- cpm(calcNormFactors(DGEList(CS)), log = TRUE)               # facilities corrected (04)
nih0 <- cpm(calcNormFactors(DGEList(RW)), log = TRUE)               # nothing corrected
idx  <- symbol_rows(nih1, cs_sym)
nih1 <- nih1[idx, , drop = FALSE]; nih0 <- nih0[idx, , drop = FALSE]
rownames(nih1) <- rownames(nih0) <- cs_sym[idx]
print(table(RGM_diagnosis = s$diagnosis))
c(RGM_samples = ncol(nih1), genes_expressed = sum(expressed), genes_by_symbol = nrow(nih1))

---
# Part 3 · The JDM-lab samples
`4b_rpkm.csv` has RPKM by gene symbol (rows `Cohort` and `ID`, then the genes). It becomes log2(RPKM + 1) and is quantile-normalised across the JDM-lab samples. The file's own `Cohort` row says which sample is JDM and which is a healthy control (matched with `HC_PATTERN` and `JDM_PATTERN` in Part 1; if the Cohort row holds no healthy controls, the sample ID text is tried instead). A table printed below shows exactly how every label was read. No key sheet is needed.

**Samples dropped, with the reason written to `excluded_samples.csv`:** the label is neither JDM nor HC; the **same library** as an RGM sample (its ID ends an RGM sample name); the **same person** as an RGM patient (`DROP_SAME_PERSON`; only possible if you give the optional key sheet in `JDM_KEY`). This is the same logic as notebook 07.

In [ ]:
raw <- read.csv(JDM_RPKM, check.names = FALSE, colClasses = "character")
gcol <- colnames(raw)[1]
is_meta <- raw[[gcol]] %in% c("Cohort", "ID")
ids <- unlist(raw[raw[[gcol]] == "ID", -1], use.names = FALSE)
cohort <- if ("Cohort" %in% raw[[gcol]]) unlist(raw[raw[[gcol]] == "Cohort", -1], use.names = FALSE) else rep("", length(ids))
cohort[is.na(cohort)] <- ""
E <- as.matrix(raw[!is_meta, -1]); storage.mode(E) <- "numeric"
rownames(E) <- raw[[gcol]][!is_meta]; colnames(E) <- ids
E <- log2(E + 1)
if (QUANTILE_NORMALISE) { full <- rowSums(is.na(E)) == 0; E[full, ] <- normalizeQuantiles(E[full, , drop = FALSE]) }
E <- E[symbol_rows(E, rownames(E)), , drop = FALSE]

# who is JDM and who is healthy: read from the file itself
clean_label <- function(x) trimws(sub("\\.\\.\\.[0-9]+$", "", x))     # R adds ...2-style suffixes to repeated labels
classify <- function(x) ifelse(grepl(HC_PATTERN, x, ignore.case = TRUE), "HC",
                        ifelse(grepl(JDM_PATTERN, x, ignore.case = TRUE), "JDM", NA))
group <- classify(clean_label(cohort))
if (!any(group == "HC", na.rm = TRUE)) group <- classify(clean_label(ids))    # no healthy controls in the Cohort row: try the ID text
cat("How the labels in 4b_rpkm.csv (Cohort row) were read:\n")
print(table(label_in_file = ifelse(cohort == "", "(blank)", clean_label(cohort)), classified_as = ifelse(is.na(group), "NEITHER", group)))
if (!any(group == "HC", na.rm = TRUE) || !any(group == "JDM", na.rm = TRUE))
  stop("Could not find both healthy controls and JDM in 4b_rpkm.csv (Cohort row or IDs). Look at the table above and change HC_PATTERN / JDM_PATTERN in Part 1 to match the labels.")
jm <- data.frame(sample = colnames(E), group = group, cohort = ifelse(cohort == "", "blank", clean_label(cohort)), stringsAsFactors = FALSE)

same_lib <- vapply(jm$sample, function(i) any(endsWith(s_all$sample_id, paste0("-", i))), logical(1))
pid_nih  <- rep("", nrow(jm))
if (nzchar(JDM_KEY)) {
  key <- read.csv(JDM_KEY, check.names = FALSE, colClasses = "character")
  colnames(key) <- trimws(colnames(key)); key <- key[!duplicated(key$ID), ]
  kk <- key[match(jm$sample, key$ID), , drop = FALSE]
  if ("NIH Enroll ID" %in% colnames(kk)) pid_nih <- norm_id(kk[["NIH Enroll ID"]])
} else cat("No key sheet given: 'same person' cannot be checked, only 'same library'.\n")
same_person <- DROP_SAME_PERSON & pid_nih != "" & pid_nih %in% s_all$patient_key
keep_j <- !is.na(jm$group) & !same_lib & !same_person
why_j  <- ifelse(is.na(jm$group), "JDM lab: label is neither JDM nor HC",
          ifelse(same_lib, "JDM lab: same library as an RGM sample",
          ifelse(same_person, "JDM lab: same person as an RGM patient", "")))
excluded <- data.frame(sample_id = jm$sample[!keep_j], source = rep("JDM-lab", sum(!keep_j)), reason = why_j[!keep_j],
                       stringsAsFactors = FALSE)

cat("JDM-lab samples read:", ncol(E), "| kept:", sum(keep_j), "| dropped:", sum(!keep_j), "\n")
print(table(reason = why_j[!keep_j]))
E <- E[, keep_j, drop = FALSE]; jm <- jm[keep_j, , drop = FALSE]
print(table(JDM_lab_group = jm$group))
Efull <- E[rowSums(is.na(E)) == 0, , drop = FALSE]
c(JDM_samples = sum(jm$group == "JDM"), JDM_lab_HC = sum(jm$group == "HC"), genes_with_a_value_in_every_sample = nrow(Efull))
cat("\nlabels (Cohort row) among the kept JDM-lab samples, by group; if there are several different labels the file may be several datasets:\n")
print(table(label = jm$cohort, group = jm$group))

---
# Part 4 · Who is in which source, and can each source be corrected?

**Table.** Rows are sources, columns are diagnoses. Read it first.

**Verdict.** A source is `OK` when it holds at least `MIN_ANCHOR` samples of an **anchor** diagnosis **and** another usable source does too. With the default `ANCHOR_ON = "HC"` the anchor is the **healthy controls only**: the one group that is the same kind of person wherever it was sequenced (the advice of the NIAID bioinformatics colleague, 2026-11-12: correction only works where comparable samples, such as healthy controls, sit in several batches). Set `ANCHOR_ON <- NULL` to also allow a disease found in two sources as an anchor, which keeps more sources but leans on the assumption that the same disease looks alike in both. Diagnosis is protected in the model either way. Leaving out one source can make another lose its only partner, so the check is repeated until nothing changes. Finally the sources that remain must form **one connected group** (every source linked to the others through shared diagnoses); a source that is linked to nobody cannot be put on the same scale.

In [ ]:
meta <- data.frame(
  sample_id = c(colnames(nih1), jm$sample),
  diagnosis = c(s$diagnosis, ifelse(jm$group == "HC", "HC", "JDM")),
  source    = c(s$batch, if (JDM_SPLIT_BY_COHORT) paste0("JDM-", make.names(jm$cohort)) else rep("JDM-lab", nrow(jm))),
  lab       = rep(c("RGM", "JDM lab"), c(ncol(nih1), nrow(jm))),
  patient_key = c(s$patient_key, rep("", nrow(jm))),
  stringsAsFactors = FALSE)
if (anyDuplicated(meta$sample_id))
  stop(sum(duplicated(meta$sample_id)), " sample name(s) appear twice across the RGM and JDM-lab data; they must be unique (check `meta$sample_id[duplicated(meta$sample_id)]`).")

tab_all <- table(source = meta$source, diagnosis = meta$diagnosis)
if (!"HC" %in% colnames(tab_all)) stop("No healthy controls ('HC') in any source: nothing can anchor the correction.")
print(tab_all)

judge <- function(keep) {                   # for each source in `keep`: which diagnoses it shares with the others
  tk <- tab_all[keep, , drop = FALSE]
  do.call(rbind, lapply(keep, function(b) {
    n_b <- tk[b, ]; others <- setdiff(keep, b)
    usable <- if (is.null(ANCHOR_ON)) colnames(tk) else intersect(ANCHOR_ON, colnames(tk))
    shared <- usable[n_b[usable] >= MIN_ANCHOR &
                           vapply(usable, function(dd) any(tk[others, dd] >= MIN_ANCHOR), logical(1))]
    data.frame(source = b, anchors = paste(shared, collapse = ", "), stringsAsFactors = FALSE)
  }))
}
largest_linked_group <- function(srcs) {    # two sources are linked if they share a diagnosis with MIN_ANCHOR+ samples in both
  if (length(srcs) < 2) return(srcs)
  cols <- if (is.null(ANCHOR_ON)) colnames(tab_all) else intersect(ANCHOR_ON, colnames(tab_all))
  linked <- outer(srcs, srcs, Vectorize(function(a, b) a != b && any(tab_all[a, cols] >= MIN_ANCHOR & tab_all[b, cols] >= MIN_ANCHOR)))
  comp <- integer(length(srcs)); k <- 0L
  for (i in seq_along(srcs)) if (comp[i] == 0L) {
    k <- k + 1L; stack <- i
    while (length(stack)) { j <- stack[1]; stack <- stack[-1]
      if (comp[j] == 0L) { comp[j] <- k; stack <- c(stack, which(linked[j, ] & comp == 0L)) } }
  }
  srcs[comp == as.integer(names(which.max(table(comp))))]
}

keep <- rownames(tab_all)[rowSums(tab_all) >= MIN_PER_SOURCE]
not_linked <- character(0)
repeat {
  if (length(keep) < 2) stop("Fewer than two usable sources are left; nothing can be put on one scale. Look at the table above.")
  v  <- judge(keep)
  ok <- v$source[v$anchors != ""]                 # sources that share a diagnosis with another usable source
  ok2 <- largest_linked_group(ok)                 # ... and that are all linked into one group
  not_linked <- union(not_linked, setdiff(ok, ok2))
  if (length(ok2) == length(keep)) break
  keep <- ok2
}

verdicts <- do.call(rbind, lapply(rownames(tab_all), function(b) {
  n_b <- tab_all[b, ]
  status <- if (sum(n_b) < MIN_PER_SOURCE) "TOO SMALL"
            else if (b %in% not_linked) "NOT LINKED TO THE OTHER SOURCES"
            else if (b %in% keep) "OK" else "CANNOT BE CORRECTED"
  anch <- if (b %in% keep) v$anchors[v$source == b] else ""
  data.frame(source = b, samples = sum(n_b), HC = unname(n_b["HC"]),
             diagnoses = paste0(names(n_b)[n_b > 0], " (", n_b[n_b > 0], ")", collapse = ", "),
             verdict = status, shift_is_measured_from = anch,
             what_it_means = switch(status,
               "OK" = paste0("its shift can be measured from: ", anch),
               "TOO SMALL" = paste0("fewer than ", MIN_PER_SOURCE, " samples"),
               "NOT LINKED TO THE OTHER SOURCES" = "shares no diagnosis with the main group of sources",
               paste0("no diagnosis here (", MIN_ANCHOR, "+ samples) is also found in another usable source, so source and disease cannot be told apart")),
             stringsAsFactors = FALSE) }))
rownames(verdicts) <- NULL
print(verdicts[, c("source", "samples", "HC", "verdict", "what_it_means")], right = FALSE)

left_out <- verdicts$source[verdicts$verdict != "OK"]
if (length(left_out)) {
  cat("\nLEFT OUT of the combined table (", sum(verdicts$samples[verdicts$verdict != "OK"]), " samples): ",
      paste(left_out, collapse = ", "), "\n", sep = "")
  cat("They are not wrong, they just cannot be placed on the common scale. Analyse them against their own healthy controls (notebooks 05 and 07).\n")
} else cat("\nEvery source can be corrected.\n")
if (length(left_out)) {
  gone <- meta$source %in% left_out
  excluded <- rbind(excluded, data.frame(sample_id = meta$sample_id[gone], source = meta$source[gone],
                    reason = paste0("source verdict: ", verdicts$verdict[match(meta$source[gone], verdicts$source)])))
}
sel_samp <- meta$source %in% keep
cat("\nsources used:", paste(keep, collapse = ", "), "\nsamples used:", sum(sel_samp), "of", nrow(meta), "\n")

---
# Part 5 · Correct

1. Genes present in both datasets (matched by symbol) and, inside **every** source, not constant (a gene with no variation inside a source cannot be fitted).
2. **ComBat** (Johnson WE, Li C, Rabinovic A. *Biostatistics* 2007;8:118-127): for each gene, estimates how much each source shifts and stretches it, borrowing strength across genes (empirical Bayes), and removes that. `mod` is the **diagnosis**, so differences between diagnoses are *not* treated as batch.
3. Before fitting, the notebook checks the design is solvable (no source is exactly a diagnosis). If it were, it stops and says so.

The three matrices kept for the proof: **`X0`** nothing corrected; **`X1`** facilities corrected (notebook 04); **`Xc`** facilities + JDM lab corrected (the result).

In [ ]:
gs   <- intersect(rownames(nih1), rownames(Efull))
cols <- which(sel_samp)
X0 <- cbind(nih0[gs, , drop = FALSE], Efull[gs, , drop = FALSE])[, cols, drop = FALSE]
X1 <- cbind(nih1[gs, , drop = FALSE], Efull[gs, , drop = FALSE])[, cols, drop = FALSE]
m  <- meta[cols, ]
stopifnot(identical(colnames(X1), m$sample_id), identical(colnames(X0), m$sample_id))
src <- factor(m$source)
grp <- factor(m$diagnosis, levels = c("HC", setdiff(sort(unique(m$diagnosis)), "HC")))

vmat <- sapply(split(seq_len(ncol(X1)), src), function(ix) apply(X1[, ix, drop = FALSE], 1, var))
ok_g <- apply(vmat, 1, function(x) all(!is.na(x) & x > 0))
X0 <- X0[ok_g, , drop = FALSE]; X1 <- X1[ok_g, , drop = FALSE]
print(c(genes_in_both_datasets = length(gs), genes_constant_inside_a_source_removed = sum(!ok_g), genes_corrected = nrow(X1)))

mod <- model.matrix(~ grp)
des <- cbind(mod, model.matrix(~ src)[, -1, drop = FALSE])
if (qr(des)$rank < ncol(des))
  stop("A source cannot be separated from the diagnoses (rank problem). Part 4 should have caught this: ",
       "check the table above for a source that holds exactly one diagnosis.")
Xc <- ComBat(dat = X1, batch = src, mod = mod, par.prior = TRUE, ref.batch = REF_BATCH)
c(genes = nrow(Xc), samples = ncol(Xc), sources = nlevels(src), missing = sum(is.na(Xc)))

---
# Part 6 · Proof it worked

## 6a · How much of the main patterns is "source"?
PCA is a 2-D map where similar samples sit close together; PC1, PC2, PC3 are its three strongest directions. For each, **R²** says how much of that direction is explained by *source* and by *diagnosis* (0 = none, 1 = all).

**What good looks like:** the *source* rows fall a lot from stage 1 to stage 3, the *diagnosis* rows stay the same or rise. If *diagnosis* falls with *source*, the correction removed disease signal too.

Triangles are healthy controls. After correction the healthy controls of every source should sit together, and disease samples should separate from them rather than from each other by source.

In [ ]:
vg <- order(apply(X0, 1, var), decreasing = TRUE)[seq_len(min(5000, nrow(X0)))]
stages <- list("1 uncorrected" = X0,
               "2 facilities corrected (notebook 04)" = X1,
               "3 facilities + JDM lab corrected (this notebook)" = Xc)
pcs <- lapply(stages, function(X) prcomp(t(X[vg, , drop = FALSE])))
row_of <- function(nm, fac, f) { r <- r2_of(pcs[[nm]], f); data.frame(stage = nm, factor = fac, PC1 = r[1], PC2 = r[2], PC3 = r[3]) }
ve <- do.call(rbind, c(lapply(names(pcs), row_of, fac = "source", f = src),
                       lapply(names(pcs), row_of, fac = "diagnosis", f = grp)))
rownames(ve) <- NULL
print(ve, right = FALSE)
write.csv(ve, file.path(OUT_DIR, "variance_explained_by_stage.csv"), row.names = FALSE)

src_col <- setNames(hcl.colors(nlevels(src), "Dark 3"), levels(src))
grp_col <- setNames(hcl.colors(nlevels(grp), "Set 2"), levels(grp))
options(repr.plot.width = 13, repr.plot.height = 15)
par(mfrow = c(3, 2), mar = c(4, 4, 3.5, 1))
for (nm in names(pcs)) {
  p <- pcs[[nm]]; sh <- round(100 * p$sdev^2 / sum(p$sdev^2), 1)
  for (by in c("source", "diagnosis")) {
    cl <- if (by == "source") src_col else grp_col
    f  <- if (by == "source") src else grp
    plot(p$x[, 1], p$x[, 2], col = cl[as.character(f)], pch = ifelse(grp == "HC", 17, 16),
         xlab = sprintf("PC1 (%.1f%%)", sh[1]), ylab = sprintf("PC2 (%.1f%%)", sh[2]),
         main = sprintf("%s\ncoloured by %s (triangles = healthy)", nm, by), cex.main = 0.85)
    legend("topright", legend = names(cl), col = cl, pch = 16, cex = 0.6, bty = "n")
  }
}
mean_r2 <- function(nm, fac) mean(unlist(ve[ve$stage == nm & ve$factor == fac, c("PC1", "PC2")]))
nm1 <- names(pcs)[1]; nm3 <- names(pcs)[3]
s_before <- mean_r2(nm1, "source"); s_after <- mean_r2(nm3, "source")
d_before <- mean_r2(nm1, "diagnosis"); d_after <- mean_r2(nm3, "diagnosis")
r2_pass <- is.finite(s_after) && (s_after < 0.5 * s_before || s_after < 0.2) && d_after >= 0.8 * d_before
cat(sprintf("\nsource share of PC1-PC2:    %.2f before -> %.2f after\ndiagnosis share of PC1-PC2: %.2f before -> %.2f after\n=> %s\n",
            s_before, s_after, d_before, d_after,
            if (r2_pass) "PASS: source fell, diagnosis kept" else "CHECK: source did not fall enough, or diagnosis fell too (read the plots)"))

## 6b · The interferon genes must survive (positive control) and the NF-κB genes must not move (negative control)

This is the check Anne's repo used (notebook 49) and the one that exposed a correction that looked fine on a PCA and had in fact erased the biology.

- **IFN_28**: the 28 interferon response genes. In CANDLE, SAVI, AGS, UAID and JDM they are expected **up** relative to healthy.
- **NFkB_11**: 11 genes controlled by NF-κB but not by interferon. They are **not** expected to follow the interferon diseases, so a big change here after correction is a warning of distortion.

**How the table is built.** Each sample gets a score: the mean over the genes of its z-score against healthy controls (value minus the healthy mean, divided by the healthy spread; a floor of 0.1 on the spread). Then, per diagnosis, the median score minus the median of the healthy controls:
- `within_source_effect`: measured **inside each source against that source's own healthy controls**, using the uncorrected values. A shift cannot fake this, so it is the fair reference. (If a disease is in several sources, the median across them.)
- `pooled_before`: all sources against **all** healthy controls pooled, uncorrected. This shows how wrong uncorrected pooling is.
- `pooled_after`: the same, on the corrected values. This is what any downstream plot will show.

**PASS** (IFN_28 only): `within_source_effect` is at least `IFN_MIN_SIGNAL`, and `pooled_after` keeps at least `PASS_FRACTION` of it.

In [ ]:
irg <- read_set("ifn-response-genes-28.txt"); nfk <- read_set("nfkb-only-control-11-dejesus2020.txt")
gene_sets <- list(IFN_28 = irg, NFkB_11 = nfk)
gene_sets <- gene_sets[!vapply(gene_sets, is.null, logical(1))]
if (!length(gene_sets)) stop("Gene lists not found in ", GENE_SET_DIR, ": copy the repo's genes/ folder there (or fix GENE_SET_DIR in Part 1).")
for (nm in names(gene_sets)) cat(nm, ": found", length(intersect(gene_sets[[nm]], rownames(Xc))), "of", length(gene_sets[[nm]]), "genes\n")

zscore <- function(X, genes, ref_cols) {
  g <- intersect(genes, rownames(X)); H <- X[g, ref_cols, drop = FALSE]
  Z <- (X[g, , drop = FALSE] - rowMeans(H)) / pmax(apply(H, 1, sd), 0.1)
  colMeans(Z)
}
delta <- function(sc, in_d, in_h) median(sc[in_d]) - median(sc[in_h])
is_hc <- grp == "HC"
diseases <- setdiff(levels(grp), "HC")
usable_src <- function(dd) levels(src)[vapply(levels(src), function(b)
  sum(src == b & grp == dd) >= MIN_ANCHOR && sum(src == b & is_hc) >= MIN_ANCHOR, logical(1))]

sig <- do.call(rbind, lapply(names(gene_sets), function(set_nm) {
  genes <- gene_sets[[set_nm]]
  p0 <- zscore(X0, genes, which(is_hc)); p3 <- zscore(Xc, genes, which(is_hc))
  do.call(rbind, lapply(diseases, function(dd) {
    us <- usable_src(dd)
    within <- if (length(us)) median(vapply(us, function(b) {
      cb <- which(src == b); sc <- zscore(X0[, cb, drop = FALSE], genes, which(is_hc[cb]))
      delta(sc, grp[cb] == dd, is_hc[cb]) }, numeric(1))) else NA_real_
    data.frame(set = set_nm, diagnosis = dd, n = sum(grp == dd), sources_with_disease_and_HC = length(us),
               within_source_effect = round(within, 2),
               pooled_before = round(delta(p0, grp == dd, is_hc), 2),
               pooled_after  = round(delta(p3, grp == dd, is_hc), 2), stringsAsFactors = FALSE)
  }))
}))
sig$verdict <- ifelse(sig$set != "IFN_28", "information only: should stay near 0",
  ifelse(is.na(sig$within_source_effect), "no reference: this disease is never in a source that also has healthy controls",
  ifelse(sig$within_source_effect < IFN_MIN_SIGNAL, "no clear interferon signal to protect",
  ifelse(sig$pooled_after >= PASS_FRACTION * sig$within_source_effect, "PASS: interferon signal kept", "FAIL: interferon signal lost"))))
rownames(sig) <- NULL
print(sig, right = FALSE)
write.csv(sig, file.path(OUT_DIR, "checks_signal_genes.csv"), row.names = FALSE)

options(repr.plot.width = 13, repr.plot.height = 9)
par(mfrow = c(2, 1), mar = c(9, 4, 3, 1))
ord <- levels(factor(paste(grp, src, sep = " | ")))
for (nm in c("1 uncorrected", "3 facilities + JDM lab corrected (this notebook)")) {
  X <- if (nm == "1 uncorrected") X0 else Xc
  sc <- zscore(X, gene_sets[[1]], which(is_hc))
  boxplot(sc ~ factor(paste(grp, src, sep = " | "), levels = ord), las = 2, xlab = "", ylab = paste0(names(gene_sets)[1], " score (z against all healthy)"),
          col = ifelse(grepl("^HC", ord), "grey80", "#7570B3"), main = paste(nm, ": ", names(gene_sets)[1], " score by diagnosis and source"), cex.axis = 0.65, cex.main = 0.9)
  abline(h = 0, lty = 2, col = "grey50")
}

## 6c · A disease that sits in two sources must look the same in both

The two checks above could be fooled: after ComBat the healthy controls of every source line up **by construction**, and a disease found in only one source has nothing to be compared with. This check cannot be fooled that way. Where one diagnosis has `MIN_ANCHOR`+ samples **and** healthy controls in two sources, take each source's fingerprint of that disease (per gene: disease mean minus healthy mean) and correlate the two (Spearman, over all genes):
- `rho_reference`: each fingerprint measured against **its own source's** healthy controls, uncorrected (the fair reference);
- `rho_after`: fingerprints from the corrected values, against **all** healthy controls pooled.

**PASS** when `rho_after` is not more than 0.1 below `rho_reference`. If no disease sits in two sources the notebook says so; then 6a and 6b are the only evidence, and that is worth knowing.

In [ ]:
shared <- list()
for (dd in diseases) {
  us <- usable_src(dd)
  if (length(us) < 2) next
  fc_ref <- function(b) rowMeans(X0[, src == b & grp == dd, drop = FALSE]) - rowMeans(X0[, src == b & is_hc, drop = FALSE])
  fc_aft <- function(b) rowMeans(Xc[, src == b & grp == dd, drop = FALSE]) - rowMeans(Xc[, is_hc, drop = FALSE])
  for (pr in combn(us, 2, simplify = FALSE)) {
    rr <- cor(fc_ref(pr[1]), fc_ref(pr[2]), method = "spearman")
    ra <- cor(fc_aft(pr[1]), fc_aft(pr[2]), method = "spearman")
    shared[[length(shared) + 1]] <- data.frame(diagnosis = dd, source_a = pr[1], source_b = pr[2],
      n_a = sum(src == pr[1] & grp == dd), n_b = sum(src == pr[2] & grp == dd),
      rho_reference = round(rr, 2), rho_after = round(ra, 2),
      verdict = ifelse(ra >= rr - 0.1, "PASS", "CHECK: fingerprints agree less after correction"), stringsAsFactors = FALSE)
  }
}
if (length(shared)) {
  shared <- do.call(rbind, shared); rownames(shared) <- NULL
  print(shared, right = FALSE)
} else {
  shared <- data.frame(diagnosis = character(), source_a = character(), source_b = character(), n_a = integer(), n_b = integer(),
                       rho_reference = numeric(), rho_after = numeric(), verdict = character())
  cat("No disease has samples and healthy controls in two or more sources, so this check cannot be run.\n",
      "That is a property of the data, not a failure: the correction rests on 6a and 6b only.\n", sep = "")
}
write.csv(shared, file.path(OUT_DIR, "checks_shared_disease.csv"), row.names = FALSE)

---
# Part 7 · Save and summary

In [ ]:
write.csv(data.frame(gene = rownames(Xc), Xc, check.names = FALSE), file.path(OUT_DIR, "corrected_log2_all_sources.csv"), row.names = FALSE)
write.csv(m[, c("sample_id", "diagnosis", "source", "lab", "patient_key")], file.path(OUT_DIR, "corrected_samples.csv"), row.names = FALSE)
write.csv(as.data.frame.matrix(tab_all), file.path(OUT_DIR, "source_by_diagnosis.csv"))
write.csv(verdicts, file.path(OUT_DIR, "source_verdicts.csv"), row.names = FALSE)
write.csv(excluded, file.path(OUT_DIR, "excluded_samples.csv"), row.names = FALSE)

n_pass <- sum(grepl("^PASS", sig$verdict)); n_fail <- sum(grepl("^FAIL", sig$verdict))
sh_pass <- sum(shared$verdict == "PASS"); sh_chk <- sum(shared$verdict != "PASS")
log_lines <- c("# All-sources correction log (notebook 04b)", paste0("Generated ", format(Sys.time(), "%Y-%m-%d %H:%M"), "\n"),
         "## Settings", paste0("USE_ONE_PER_PATIENT=", USE_ONE_PER_PATIENT, "; MIN_PER_SOURCE=", MIN_PER_SOURCE, "; MIN_ANCHOR=", MIN_ANCHOR,
                               "; IFN_MIN_SIGNAL=", IFN_MIN_SIGNAL, "; PASS_FRACTION=", PASS_FRACTION),
         "", "## Source verdicts", "```", capture.output(print(verdicts[, c("source", "samples", "HC", "verdict")], right = FALSE)), "```", "",
         "## Variance explained", "```", capture.output(print(ve, right = FALSE)), "```", "",
         "## Interferon / NF-kB checks", "```", capture.output(print(sig, right = FALSE)), "```", "",
         "## Shared-disease check", "```", capture.output(print(shared, right = FALSE)), "```")
writeLines(log_lines, file.path(OUT_DIR, "PROCESSING_LOG_all_sources.md"))

cat("=========================== SUMMARY ===========================\n")
cat("sources used (", length(keep), "): ", paste(keep, collapse = ", "), "\n", sep = "")
cat("sources left out (", length(left_out), "): ", if (length(left_out)) paste(left_out, collapse = ", ") else "none", "\n", sep = "")
cat("samples in the corrected table:", ncol(Xc), "| genes:", nrow(Xc), "\n")
cat("6a source falls, diagnosis kept:        ", if (r2_pass) "PASS" else "CHECK", "\n")
cat("6b interferon signal kept:              ", n_pass, "PASS,", n_fail, "FAIL\n")
cat("6c shared-disease fingerprints agree:   ", if (nrow(shared)) paste(sh_pass, "PASS,", sh_chk, "CHECK") else "not testable with these data", "\n")
if (r2_pass && n_fail == 0 && sh_chk == 0) {
  cat("\nAll runnable checks passed. Use corrected_log2_all_sources.csv for plots and clustering.\n")
} else {
  cat("\nAt least one check did not pass. Do not use the corrected table until you have looked at it (see 'What to check' below).\n")
}
cat("\nfiles written to", OUT_DIR, ":\n"); print(list.files(OUT_DIR))

---
## What to check
1. **Part 4 verdict table.** Which sources were left out, and why? A left-out source is not an error: it cannot be put on the common scale from these data. If you need it in, the fix is on the sequencing side (healthy controls sequenced at that facility).
2. **6a.** *source* should fall, *diagnosis* should hold. The coloured PCA should no longer show one cluster per source.
3. **6b.** Every `IFN_28` row should say `PASS` (or "no clear interferon signal to protect", meaning the disease has no interferon effect to lose). A `FAIL` means the correction erased interferon signal: do not use the table. The `NFkB_11` rows should stay small.
4. **6c.** If it ran, `rho_after` close to `rho_reference` means a disease looks the same in two places after correction.
5. **`excluded_samples.csv`.** Every sample that is not in the corrected table, with its reason.

## Limits (what this does not prove)
- **Healthy controls are the ruler.** The method assumes the healthy controls of one source are the same kind of healthy blood as those of another. The JDM-lab files are whole blood only because you said so; nothing in the files shows it.
- **Is the JDM file one dataset?** The JDM data is reportedly a collection of datasets. If so, treating it as one source (the default) hides batch effects *inside* it. Part 3 prints the labels of the Cohort row; if they name different datasets and each holds healthy controls, set `JDM_SPLIT_BY_COHORT <- TRUE` so each becomes its own source. If the labels are just JDM and HC, splitting would confound them, so leave it `FALSE`.
- **A disease found in one source only** (JDM) is placed using the shift measured on healthy controls and assumed to hold for the disease too. 6c cannot test this. Treat JDM-vs-RGM-disease gaps in the corrected table as suggestive; notebook 07 (each disease against its own lab's healthy controls) is the version that does not rest on this assumption.
- **Few healthy controls in a source** makes its shift noisy. The table shows how many each source has (`HC`).
- **Do not run statistical tests on the corrected values.** ComBat makes samples look more alike than they are, which makes tests too confident. Tests: notebook 05 (raw counts, batch as a model term) or notebook 07 (each lab's own comparison).

## References
1. Johnson WE, Li C, Rabinovic A. Adjusting batch effects in microarray expression data using empirical Bayes methods. *Biostatistics* 2007;8:118-127.
2. Zhang Y, Parmigiani G, Johnson WE. ComBat-seq: batch effect adjustment for RNA-seq count data. *NAR Genom Bioinform* 2020;2:lqaa078.
3. Leek JT et al. The sva package for removing batch effects and other unwanted variation in high-throughput experiments. *Bioinformatics* 2012;28:882-883.
4. Robinson MD, McCarthy DJ, Smyth GK. edgeR. *Bioinformatics* 2010;26:139-140. Ritchie ME et al. limma. *Nucleic Acids Res* 2015;43:e47.
5. Kim H et al. *J Interferon Cytokine Res* 2018;38:171-185; de Jesus AA et al. *J Clin Invest* 2020;130:1669-1682 (interferon and NF-κB gene sets).
6. Risso D, Ngai J, Speed TP, Dudoit S. Normalization of RNA-seq data using factor analysis of control genes or samples. *Nat Biotechnol* 2014;32:896-902 (why control-gene correction was not used: see Anne's notebook 49).